# 28 · Agent-to-Agent protocol

A2A exposes an agent as a service with an agent card. This optional lab runs a loopback A2A server and calls it through RemoteA2aAgent.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** Optional: remote Lite calls on the same VM. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


The current VM is missing the `a2a` dependency. Install once in the existing kernel if desired: `%pip install 'google-adk[a2a]==2.10.0'`. No remote service is deployed.


In [ ]:
import importlib.util, subprocess, asyncio, socket
if importlib.util.find_spec("a2a") is None:
    print("Optional dependency missing: install google-adk[a2a]==2.10.0, then rerun.")
else:
    import httpx
    from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", 0))
        port = probe.getsockname()[1]
    log_path = ROOT / ".runtime" / "a2a.log"
    log_path.parent.mkdir(exist_ok=True)
    with log_path.open("w") as log:
        process = subprocess.Popen([sys.executable, str(ROOT / "a2a_server.py"), str(port)], stdout=log, stderr=log, cwd=ROOT)
        try:
            card_url = f"http://127.0.0.1:{port}/.well-known/agent-card.json"
            async with httpx.AsyncClient(trust_env=False) as client:
                for attempt in range(60):
                    if process.poll() is not None:
                        raise RuntimeError(log_path.read_text())
                    try:
                        response = await client.get(card_url)
                        if response.status_code == 200:
                            break
                    except httpx.ConnectError:
                        pass
                    await asyncio.sleep(0.5)
                else:
                    raise TimeoutError("A2A server did not start; inspect .runtime/a2a.log")
                print("Agent card:", response.json()["name"])
            remote = RemoteA2aAgent(name="remote_orders", agent_card=card_url)
            lab = await make_lab(remote)
            await ask(lab, "What is the status of O1001?")
        finally:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()


## Try it

Inspect the agent card's capabilities. Compare A2A agent delegation with MCP tool calls.


## Reference

[Official ADK documentation](https://adk.dev/a2a/). Shared configuration: `config.json`. No environment activation or login cells are needed.
